<a href="https://colab.research.google.com/github/inoue0426/llm-tuning-playground/blob/main/notebooks/06_ctd_reasoning_sft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 - CTD evidence-chain reasoning with LoRA SFT

This notebook switches the fine-tuning task from generic preference alignment to **biomedical reasoning grounded in the Comparative Toxicogenomics Database (CTD)**.

We construct a simple two-hop task from curated CTD facts:

`chemical -> gene -> disease`

The model is trained to answer with the disease and an explicit evidence chain. The reasoning text is **structured evidence reasoning generated from database relations**, not free-form hidden chain-of-thought.

CTD provides curated chemical-gene and gene-disease relationships and is designed to support chemical-gene-disease network reasoning. The current notebook uses local copies of CTD files so the source data is not redistributed in this public repository.

In [ ]:
!pip -q install -U transformers datasets peft accelerate pandas scikit-learn

import ast, gzip, os, random, re
import pandas as pd
import torch
from datasets import Dataset

print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

## 1. Upload CTD source files

Upload these public CTD files to the Colab runtime:

- `CTD_chem_gene_ixns.tsv.gz`
- `CTD_curated_genes_diseases.tsv.gz` (preferred) or `CTD_genes_diseases.tsv.gz`

The files remain in `/content` and are not committed to GitHub. CTD publishes downloadable TSV/GZ data and curated chemical-gene and gene-disease relationships.

In [ ]:
from google.colab import files
uploaded = files.upload()

CHEM_GENE = '/content/CTD_chem_gene_ixns.tsv.gz'
GENE_DISEASE = '/content/CTD_curated_genes_diseases.tsv.gz'
if not os.path.exists(GENE_DISEASE):
    GENE_DISEASE = '/content/CTD_genes_diseases.tsv.gz'

if not os.path.exists(CHEM_GENE):
    raise FileNotFoundError('Upload CTD_chem_gene_ixns.tsv.gz')
if not os.path.exists(GENE_DISEASE):
    raise FileNotFoundError('Upload CTD_curated_genes_diseases.tsv.gz or CTD_genes_diseases.tsv.gz')
print('Chemical-gene:', CHEM_GENE)
print('Gene-disease:', GENE_DISEASE)

## 2. Build a two-hop CTD graph slice

We keep human chemical-gene records where possible and sample a manageable graph slice. Then we stream through the gene-disease file and retain only genes observed in the chemical-gene slice.

In [ ]:
# Read only the columns needed for the reasoning task.
chem_cols = ['ChemicalName','ChemicalID','GeneSymbol','GeneID','OrganismID','Interaction','InteractionActions','PubMedIDs']
chem_chunks = []
for chunk in pd.read_csv(CHEM_GENE, sep='\t', comment='#', compression='gzip', dtype=str, usecols=lambda c: c in chem_cols, chunksize=100000):
    if 'OrganismID' in chunk.columns:
        human = chunk[chunk['OrganismID'].fillna('').str.contains('9606', regex=False)]
    else:
        human = chunk
    chem_chunks.append(human)
    if sum(len(x) for x in chem_chunks) >= 150000:
        break
chem_gene = pd.concat(chem_chunks, ignore_index=True).dropna(subset=['ChemicalName','GeneSymbol'])
chem_gene = chem_gene.drop_duplicates(subset=['ChemicalName','GeneSymbol','InteractionActions'])
genes_needed = set(chem_gene['GeneSymbol'].astype(str))
print('Chemical-gene rows:', len(chem_gene))
print('Unique genes:', len(genes_needed))

# Stream gene-disease associations and keep only genes in the chemical-gene slice.
gd_chunks = []
gd_candidates = ['GeneSymbol','GeneID','DiseaseName','DiseaseID','DirectEvidence','PubMedIDs']
for chunk in pd.read_csv(GENE_DISEASE, sep='\t', comment='#', compression='gzip', dtype=str, usecols=lambda c: c in gd_candidates, chunksize=100000):
    if 'GeneSymbol' not in chunk.columns or 'DiseaseName' not in chunk.columns:
        raise ValueError(f'Unexpected gene-disease columns: {list(chunk.columns)}')
    hit = chunk[chunk['GeneSymbol'].isin(genes_needed)]
    if len(hit):
        gd_chunks.append(hit)
gene_disease = pd.concat(gd_chunks, ignore_index=True).dropna(subset=['GeneSymbol','DiseaseName'])
gene_disease = gene_disease.drop_duplicates(subset=['GeneSymbol','DiseaseName'])
print('Gene-disease rows:', len(gene_disease))
print(gene_disease.head())

## 3. Construct evidence-chain reasoning examples

Each example is a two-hop path: a curated chemical-gene relation followed by a curated gene-disease association. We cap the number of diseases per gene and the number of examples per chemical so the task stays balanced.

In [ ]:
gd_map = {}
for _, r in gene_disease.iterrows():
    g = str(r['GeneSymbol']).strip()
    d = str(r['DiseaseName']).strip()
    if g and d:
        gd_map.setdefault(g, []).append(d)

records = []
per_chemical = {}
for _, r in chem_gene.iterrows():
    chemical = str(r['ChemicalName']).strip()
    gene = str(r['GeneSymbol']).strip()
    interaction = str(r.get('InteractionActions', r.get('Interaction', 'associated with'))).strip()
    diseases = list(dict.fromkeys(gd_map.get(gene, [])))[:10]
    if not diseases:
        continue
    for disease in diseases[:3]:
        if per_chemical.get(chemical, 0) >= 3:
            break
        prompt = (
            f'Use the two curated facts below to infer a possible chemical-disease connection.\n\n'
            f'Fact 1: {chemical} has this chemical-gene relationship with {gene}: {interaction}.\n'
            f'Fact 2: {gene} is associated with the disease {disease}.\n\n'
            f'Question: Which disease is connected to {chemical} through {gene}? '
            f'Give the disease name and a concise two-step evidence chain.'
        )
        answer = (
            f'Answer: {disease}\n'
            f'Evidence chain: {chemical} -> {gene} ({interaction}); {gene} -> {disease}.\n'
            f'Conclusion: the two curated relations connect {chemical} to {disease} through {gene}.'
        )
        records.append({'chemical': chemical, 'gene': gene, 'disease': disease, 'prompt': prompt, 'answer': answer})
        per_chemical[chemical] = per_chemical.get(chemical, 0) + 1

random.Random(42).shuffle(records)
records = records[:8000]
print('Reasoning examples:', len(records))
print(records[0])

In [ ]:
# Chemical-disjoint split: evaluation chemicals are unseen during training.
chemicals = sorted({r['chemical'] for r in records})
random.Random(42).shuffle(chemicals)
cut = int(0.8 * len(chemicals))
train_chems = set(chemicals[:cut])
train_records = [r for r in records if r['chemical'] in train_chems]
eval_records = [r for r in records if r['chemical'] not in train_chems]

print('Train:', len(train_records))
print('Eval:', len(eval_records))
print('Train chemicals:', len(train_chems))
print('Eval chemicals:', len(set(r['chemical'] for r in eval_records)))

## 4. LoRA SFT

We use Qwen2.5-0.5B-Instruct again, but this time the target behavior is explicit evidence-chain reasoning rather than generic answer style.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype).cuda()

lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=['q_proj','k_proj','v_proj','o_proj'], bias='none', task_type='CAUSAL_LM')
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
def tokenize_reasoning_example(ex, max_length=512):
    user = [{'role': 'user', 'content': ex['prompt']}]
    full = user + [{'role': 'assistant', 'content': ex['answer']}]
    prompt_ids = tokenizer.apply_chat_template(user, tokenize=True, add_generation_prompt=True)['input_ids']
    full_ids = tokenizer.apply_chat_template(full, tokenize=True, add_generation_prompt=False)['input_ids']
    full_ids = full_ids[:max_length]
    labels = [-100] * min(len(prompt_ids), len(full_ids)) + full_ids[min(len(prompt_ids), len(full_ids)):]
    return {'input_ids': full_ids, 'attention_mask': [1] * len(full_ids), 'labels': labels}

train_ds = Dataset.from_list(train_records).map(tokenize_reasoning_example, remove_columns=Dataset.from_list(train_records).column_names)
eval_ds = Dataset.from_list(eval_records).map(tokenize_reasoning_example, remove_columns=Dataset.from_list(eval_records).column_names)
print(train_ds[0].keys())

In [ ]:
def collate(features):
    max_len = max(len(x['input_ids']) for x in features)
    input_ids = []
    attention = []
    labels = []
    for x in features:
        pad = max_len - len(x['input_ids'])
        input_ids.append(x['input_ids'] + [tokenizer.pad_token_id] * pad)
        attention.append(x['attention_mask'] + [0] * pad)
        labels.append(x['labels'] + [-100] * pad)
    return {
        'input_ids': torch.tensor(input_ids, dtype=torch.long),
        'attention_mask': torch.tensor(attention, dtype=torch.long),
        'labels': torch.tensor(labels, dtype=torch.long),
    }

args = TrainingArguments(output_dir='./outputs/ctd-reasoning', per_device_train_batch_size=2, gradient_accumulation_steps=8, learning_rate=2e-4, num_train_epochs=2, logging_steps=20, save_strategy='no', report_to='none', fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported())
trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collate)
trainer.train()

## 5. Before / after qualitative evaluation

For a strict quantitative experiment, evaluate disease-name extraction and evidence-chain validity on the chemical-disjoint set. The generated reasoning should stay tied to the two supplied CTD facts.

In [ ]:
def generate_reasoning(model, ex, max_new_tokens=120):
    messages = [{'role':'user','content': ex['prompt']}]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors='pt', return_dict=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=True)

print('AFTER SFT')
for ex in eval_records[:5]:
    print('=' * 80)
    print('QUESTION:', ex['prompt'])
    print('GOLD:', ex['answer'])
    print('MODEL:', generate_reasoning(model, ex))


## What this experiment is actually testing

1. The model receives explicit CTD evidence.
2. The target is a two-hop chemical -> gene -> disease inference.
3. The SFT target is a structured evidence chain rather than unsupported biomedical prose.
4. The split is chemical-disjoint, so evaluation chemicals are unseen during training.

This is a better fit for biomedical reasoning than the earlier DrugBank -> UniProt open-generation task. It is still a synthetic reasoning task because the answer and evidence explanation are programmatically assembled from CTD relations.